# 강의 04 · 실습 2 — 체크포인트와 사람 개입 · (3) 변형

## 1. 문제상황

- 여행사 고객센터는 항공권 예약 변경 문의를 메일로 받습니다.
- 변경 수수료를 받을지 면제할지는 예약 조건과 고객 사정을 보고 담당자가 정합니다.
- 답장을 쓰는 사람이 그 방침을 모른 채 초안을 쓰면, 담당자가 방침을 정한 뒤 초안을 처음부터 다시 써야 합니다.
- 같은 고객이 여러 통을 보내면 앞선 메일을 다시 읽어야 하고, 처리 도중 프로그램이 꺼지면 담당자가 정한 방침도 함께 사라집니다.

## 2. 문제와 목표

- **문제**: 초안을 쓴 뒤에 담당자 방침이 정해지므로 초안을 다시 써야 하고, 앞선 메일을 사람이 다시 읽어야 하고, 프로그램이 꺼지면 방침과 진행 상황이 사라집니다.
- **목표**
  - 고객 한 사람의 메일을 같은 `thread_id` 아래 이어서 받습니다.
  - 초안을 쓰기 전에 멈춰 담당자에게 처리 방침을 받고, 그 방침대로 초안을 써서 발송하는 처리 흐름을 만듭니다.
  - 노드가 끝날 때마다 진행 상황을 파일에 저장하고, 대화가 길어지면 오래된 메시지를 요약으로 바꿉니다.
- **목표 달성 여부의 판정 기준**: 같은 `thread_id`로 고객 메일 세 통을 차례로 넣었을 때,
  - 메일마다 그래프가 초안을 만들기 전에 멈추고, 담당자 방침을 넘긴 뒤에만 초안이 만들어지고 발송되며,
  - 초안에 방침의 핵심(수수료 안내, 면제 조건, 환불 가능 여부)이 들어 있고,
  - 두 번째 메일부터 대화 기록의 앞부분이 요약 메시지로 바뀌어 있고 보낸 답장이 대화 기록에 남아 있는 것을 실행 결과에서 확인합니다.
  - 담당자의 방침은 대본으로 미리 넣습니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex02_s3_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 대화 기록(`messages`), 담당자 방침(`policy`), 답장 초안(`draft`), 발송 여부(`sent`) 키 네 개를 가지는 상태를 선언합니다.
    - `messages` 키에는 `add_messages` 리듀서를 붙입니다.
2. **컨텍스트 관리 노드를 만듭니다.**
    - manage 노드는 `messages`의 길이가 `KEEP`(이 실습에서는 2) 이하이면 아무것도 바꾸지 않습니다.
    - `KEEP`을 넘으면 최근 `KEEP` 개만 원문으로 남기고, 그 앞의 메시지들을 요약 지시문 `SUMMARY_RULE`로 모델에 요약시킨 뒤, 오래된 메시지들을 `RemoveMessage`로 지우고 요약 `SystemMessage` 하나를 넣습니다.
    - 요약 지시문은 「다음 여행사 고객센터 대화를 고객이 문의한 내용과 상담원이 안내한 처리 중심으로 두 문장 이내 한국어로 요약한다. 새 정보를 지어내지 않는다.」입니다.
3. **방침 노드를 만듭니다.**
    - ask_policy 노드는 `interrupt()`로 실행을 멈추고, 질문과 마지막 고객 메일 본문(대화 기록에서 가장 최근의 `HumanMessage`)을 담당자에게 보냅니다.
    - 담당자가 준 값을 문자열로 `policy` 키에 씁니다.
4. **초안 노드를 만듭니다.**
    - draft 노드는 `policy` 키의 방침을 시스템 프롬프트에 넣고 `messages` 전체를 모델에 넣어, 마지막 고객 메일에 답하는 두 문장짜리 답장 초안을 `draft` 키에 씁니다.
5. **발송 노드를 만듭니다.**
    - send 노드는 초안을 발송하고(이 실습에서 발송은 화면 출력으로 대신합니다), 보낸 답장을 `AIMessage`로 `messages` 키에 쌓고, `sent` 키에 `True`를 씁니다.
6. **그래프에 노드를 등록합니다.**
    - 네 노드를 이름과 함께 그래프에 등록합니다.
7. **엣지를 연결합니다.**
    - START → manage → ask_policy → draft → send → END를 고정 엣지로 연결합니다.
8. **체크포인터를 장착해 컴파일합니다.**
    - SQLite 파일에 저장하는 체크포인터(`SqliteSaver`)를 열어 `compile(checkpointer=…)`에 넘깁니다.
    - 실행할 때마다 `thread_id`를 담은 설정을 함께 넘깁니다.
    - `thread_id`는 `customer-7731`입니다.
9. **그래프를 실행합니다.**
    - 같은 `thread_id`로 고객 메일 세 통을 차례로 넣습니다.
    - 메일마다 멈춘 지점(`next`)과 담당자에게 간 내용을 출력하고, 담당자 방침을 `Command(resume=…)`으로 넘긴 뒤 방침·초안·대화 기록을 출력합니다.
    - 출력 줄에는 「[멈춘 지점]」「[담당자에게 간 내용]」「[방침]」「[초안]」「[발송]」「[manage]」「[대화 기록]」 표지를 붙입니다.
    - 멈춘 지점은 `next = (노드 이름,)` 형태로 출력합니다.
    - 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 체크포인터와 `interrupt()`는 새 단계가 아니라 ⑤ 컴파일과 실행 단계의 확장입니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키를 선언합니다 | `class PolicyState(TypedDict)`, `Annotated[list, add_messages]` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def manage(state) -> dict`, `interrupt()` | 2, 3, 4, 5 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(PolicyState)`, `add_node` | 6 |
| ④ 엣지 연결 | 노드 사이의 순서를 정합니다 | `add_edge` | 7 |
| ⑤ 컴파일과 실행 | 체크포인터를 달아 컴파일하고, `thread_id`를 넘겨 실행하고, 멈춘 지점에서 이어 갑니다 | `compile(checkpointer=…)`, `invoke`, `Command(resume=…)`, `get_state` | 8, 9 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다. 체크포인트를 저장할 파일 위치도 여기서 정합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

- 체크포인트 파일은 실행할 때마다 새 임시 폴더에 만듭니다. 지난 실행의 저장 기록이 이번 실행에 섞이지 않게 하기 위해서입니다.
- 대화 기록 출력은 `show_messages(msgs)`로 합니다.

In [ ]:
import sqlite3
import tempfile
from pathlib import Path
import os

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage, SystemMessage
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Command, interrupt

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")

DB_PATH = Path(tempfile.mkdtemp(prefix="lec04_ex02_")) / "checkpoint.db"   # 체크포인트를 저장할 파일


def show_messages(msgs: list) -> None:
    """대화 기록을 메시지 종류와 앞부분 글자만 한 줄씩 출력한다."""
    for m in msgs:
        print(f"      [{type(m).__name__}] {str(m.content)[:90]}")


print("모델 준비를 마쳤습니다. 체크포인트 파일:", DB_PATH.name)


# 주어진 자료
EMAILS = [
    "다음 주 화요일 출발 항공권을 목요일로 바꾸고 싶습니다. 수수료가 있나요?",
    "회사 사정으로 바뀐 거라 수수료가 부담됩니다. 면제가 될까요?",
    "감사합니다. 그러면 아예 취소하고 환불받는 것도 가능한가요?",
]
POLICIES = [   # 메일 N에 담당자가 주는 방침 (대본)
    "변경 수수료는 규정대로 3만 원을 안내한다. 변경 희망 날짜의 좌석을 확인해 준다.",
    "회사 사정 증빙을 보내면 수수료를 면제한다. 증빙 서류 종류를 안내한다.",
    "이 운임은 환불 불가 운임이다. 취소 환불은 불가하고 날짜 변경만 가능하다고 안내한다.",
]


### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. `messages` 키에 붙인 `add_messages`가 리듀서(reducer)입니다. 노드가 `messages`에 메시지를 돌려주면 리듀서가 기존 목록 뒤에 붙이고, `RemoveMessage`를 돌려주면 같은 `id`의 메시지를 지웁니다.

In [ ]:
# 여기에 단계 ①(상태 정의)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4, 5)

- 노드는 상태를 인자로 받아 딕셔너리를 돌려주는 파이썬 함수입니다.
- `KEEP`과 `SUMMARY_RULE`은 manage 노드가 따르는 원칙입니다.
- `interrupt()`는 노드 실행을 그 자리에서 멈추고, 담당자가 준 값을 그 호출의 반환값으로 받아 이어 가는 함수입니다. 이 단에서는 draft 앞의 ask_policy 노드가 부릅니다.

In [ ]:
# 여기에 단계 ②(원칙 상수와 노드 함수 네 개 정의)를 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 6)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다.

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 7)

`add_edge`로 고정 엣지 다섯 개를 놓습니다. 조건부 엣지는 없습니다.

In [ ]:
# 여기에 단계 ④(엣지 연결)를 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 8, 9)

체크포인터를 달아 컴파일하고, 같은 `thread_id`로 메일 세 통을 실행합니다. 체크포인터 장착과 메일 세 통의 실행이 모두 이 한 단계에 속합니다.

체크포인터는 `conn = sqlite3.connect(DB_PATH, check_same_thread=False)` → `saver = SqliteSaver(conn)` → `saver.setup()` 세 줄로 엽니다. 실행 설정은 `config = {"configurable": {"thread_id": "customer-7731"}}`이고, 멈춘 지점은 `graph.get_state(config).next`, 담당자에게 간 내용은 `invoke` 반환값의 `out["__interrupt__"][0].value`에서 읽습니다.


In [ ]:
# 여기에 단계 ⑤(체크포인터 장착·컴파일, 메일 세 통의 실행: 멈춤·방침 전달·초안·발송·대화 기록 출력)를 작성합니다.

## 7. 실행 결과 확인

위 실행 결과에서 다음 세 가지를 확인합니다.

1. 메일마다 `next = ('ask_policy',)`가 출력되고, 담당자에게 간 내용에 질문과 고객 메일 본문이 들어 있습니다. `[초안]`과 `[발송]` 줄은 `Command(resume=…)` 뒤에만 출력됩니다.
2. `[방침]` 줄과 `[초안]` 줄을 나란히 읽었을 때, 방침의 핵심(수수료 면제 여부, 환불 가능 여부)이 초안에 들어 있습니다.
3. 2번 메일부터 `[manage]` 줄이 출력되고, 대화 기록에 요약 `SystemMessage`가 들어 있으며 원문은 최근 두 개만 남습니다. 보낸 답장이 `AIMessage`로 대화 기록에 남아 있습니다.